In [10]:
import requests

def search_wikipedia(query):
    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "list": "search",
            "srsearch": query,
            "format": "json"
        },
        headers={
            "User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"
        }
    )
    return response.json()

In [14]:
result

{'batchcomplete': '',
 'continue': {'sroffset': 10, 'continue': '-||'},
 'query': {'searchinfo': {'totalhits': 910,
   'suggestion': 'capivara',
   'suggestionsnippet': 'capivara'},
  'search': [{'ns': 0,
    'title': 'Capybara',
    'pageid': 6776,
    'size': 37074,
    'wordcount': 3734,
    'snippet': 'The <span class="searchmatch">capybara</span> or greater <span class="searchmatch">capybara</span> (Hydrochoerus hydrochaeris) is the largest living rodent, native to all countries in South America except Chile. It is',
    'timestamp': '2026-04-17T08:54:32Z'},
   {'ns': 0,
    'title': 'Capybara (disambiguation)',
    'pageid': 69085306,
    'size': 516,
    'wordcount': 100,
    'snippet': 'Look up <span class="searchmatch">capybara</span>\xa0or <span class="searchmatch">Capybara</span> in Wiktionary, the free dictionary. The <span class="searchmatch">capybara</span> is a giant cavy rodent native to South America. <span class="searchmatch">Capybara</span> may also refer to:',
    '

In [11]:
result = search_wikipedia("capybara")
print(result['query']['searchinfo']['totalhits'])  # Q1
print([r['title'] for r in result['query']['search']])  # Q2

910
['Capybara', 'Capybara (disambiguation)', 'Lesser capybara', 'Capybara Games', 'Hydrochoerus', 'Caviidae', 'Flow (2024 film)', 'Capybara (software)', 'Kerodon', 'Nordelta, Buenos Aires']


In [12]:
print(len(result['query']['search']))

10


In [13]:
titles = [r['title'] for r in result['query']['search']]
capybara_count = sum(1 for t in titles if 'capybara' in t.lower())
print(capybara_count)

5


In [29]:
def get_page(page_title):
    response = requests.get(
        "https://en.wikipedia.org/w/index.php",
        params={
            "action": "raw",
            "title": page_title
        },
        headers={
            "User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"
        }
    )
    return response.text

In [30]:
content = get_page("Capybara")
print(len(content))
print(content[:200])

36946
{{Short description|Largest species of rodents}}
{{Other uses}}
{{Good article}}
{{pp|small=yes}}
{{Use dmy dates|date=July 2022}}
{{Speciesbox
| status = LC
| status_system = IUCN3.1
| status_ref = <


In [31]:
from pydantic_ai import Agent
from pydantic_ai.models.gemini import GeminiModel

model = GeminiModel('gemini-2.5-flash')

wiki_agent = Agent(
    model=model,
    name='wikipedia',
    instructions="You are a helpful assistant that answers questions using Wikipedia. Use search_wikipedia to find relevant pages and get_page to fetch their content.",
    tools=[search_wikipedia, get_page]
)

/tmp/ipykernel_30535/4272120005.py:4: DeprecationWarning: Use `GoogleModel` instead. See <https://ai.pydantic.dev/models/google/> for more details.
  model = GeminiModel('gemini-2.5-flash')


In [32]:
result = await wiki_agent.run("What is this page about? https://en.wikipedia.org/wiki/Capybara")
print(result.output)

The capybara (Hydrochoerus hydrochaeris) is the largest living rodent, native to all South American countries except Chile. It is a semiaquatic herbivore, inhabiting savannas and dense forests near water bodies, feeding primarily on grasses and aquatic plants.

Capybaras are highly social, typically living in groups of 10–20 individuals, though much larger groups can form during the dry season. They are excellent swimmers and can stay submerged for up to five minutes, an ability used to evade predators like jaguars, ocelots, cougars, eagles, caimans, and anacondas.

Capybaras are not considered a threatened species, with stable populations across most of their range. They are hunted for their meat and hide in some areas, and sometimes killed by humans who perceive them as competition for livestock. In some regions, they are farmed, which helps protect their wetland habitats. They are also known to adapt well to urbanized environments and are popular in zoos and parks. In popular cultur

In [33]:
result = await wiki_agent.run("What are the main threats to capybara populations?")
print(result.output)

Capybara populations face several threats, although they are not currently considered a threatened species overall. The main threats include:

*   **Hunting:** Capybaras are hunted for their meat, pelts, and the grease from their thick fatty skin.
*   **Human conflict:** They are sometimes killed by humans who perceive them as competition for livestock grazing.
*   **Natural Predation:** In the wild, their lifespan is often cut short by predators such as jaguars, ocelots, cougars, eagles, caimans, and green anacondas.

Despite these factors, their ability to breed rapidly helps maintain a stable population throughout most of their South American range.


In [34]:
print(result.usage())

RunUsage(input_tokens=19709, cache_read_tokens=2984, output_tokens=801, details={'thoughts_tokens': 605, 'text_prompt_tokens': 19709, 'cached_content_tokens': 2984, 'text_cache_tokens': 2984}, requests=4, tool_calls=3)


In [35]:
from pydantic_ai.messages import ToolCallPart, ToolReturnPart

for msg in result.all_messages():
    for part in msg.parts:
        if hasattr(part, 'tool_name'):
            print(f"{part.part_kind}: {part.tool_name}")

tool-call: search_wikipedia
tool-return: search_wikipedia
tool-call: search_wikipedia
tool-return: search_wikipedia
tool-call: get_page
tool-return: get_page
